In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import custom CRUD module for database interactions
from crud import AnimalShelter

###########################
# Database Connection
###########################

USER = 'juan337492'
PASS = 'H3JW3qob6iP7fxF6'
HOST = 'cluster0.rvpr0.mongodb.net'
PORT = 27017
DB = 'AAC;'
COL = 'animals'
db = AnimalShelter(USER, PASS, HOST, PORT, DB, COL)


# Retrieve all animal records from the MongoDB database and convert them into a Pandas DataFrame
df = pd.DataFrame.from_records(db.read({}))

# Remove '_id' column to prevent issues with Dash DataTable (MongoDB stores it as ObjectID)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Adding in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define the layout of the dashboard
app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard - Juan Rodriguez'))),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '200px', 'width':'auto'}
            ),
             # Filter options for rescue type selection
    html.Hr(),
    html.Div([
        dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value':'Mountain'},
            {'label': 'Disaster or Individual Tracking', 'value':'Disaster'},
            {'label': 'Reset', 'value':'Reset'}
        ]
        
        )
    ]
        
# Data Table for displaying the list of animals
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
        
        row_selectable ="single",
        page_size= 10,
        page_current=0,
        selected_rows=[],
        page_action="native",
        filter_action="native",
        sort_action="native",
        
                        ),
    html.Br(),
    html.Hr(),
# Row layout for chart and geo-location map
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to update the data table based on selected rescue type
@app.callback([Output('datatable-id','data'),
              Output('datatable-id','columns'),
              Output('datatable-id','selected_rows')],
                     
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    
     # Fetch all data if "Reset" is selected
    if filter_type == 'Reset':
        df = pd.DataFrame.from_records(db.read({}))
    else:
        # Filter data by `rescue_type`
        df = pd.DataFrame.from_records(db.read({"rescue_type": filter_type}))

    # Drop `_id` column to prevent compatibility issues
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    # Prepare the data table columns and data
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data = df.to_dict('records')
    selected_rows = [0]

    return data, columns, selected_rows
    
# Callback to generate and update the pie chart
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [dcc.Graph(figure={})] #Return an empty fig if there is no data
    
    dff = pd.DataFrame.from_dict(viewData)
    
    print ("Filtered Data:")
    print(dff.head())
    df_grouped = dff.groupby(['breed']).size().reset_index(name='Count')
    
    fig = px.pie(df_grouped, values='Count', names='breed', title="Distribution of Animal Breeds")
    
    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []  # Return an empty map if there's no data

    dff = pd.DataFrame.from_dict(viewData)

    # Default to the first row if no row is selected
    row = index[0] if index else 0

    # Ensure valid row index
    if row >= len(dff):
        return []

    # Access latitude and longitude by column names
    lat = dff.at[row, 'location_lat']
    lon = dff.at[row, 'location_long']
    breed = dff.at[row, 'breed']
    rescue_type = dff.at[row, 'rescue_type']

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(f"Breed: {breed}"),
                dl.Popup([
                    html.H1("Rescue Type"),
                    html.P(rescue_type)
                ])
            ])
        ])
    ]




app.run_server(debug=True)


c:\Users\juan3\anaconda3\Lib\site-packages\dash\dash.py:579: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



Dash app running on http://127.0.0.1:8050/
